In [1]:
from pathlib import Path

import pickle
import numpy as np
import matplotlib.pyplot as plt

In [2]:
class CIFAR_10:
    metadata = None

    def __new__(cls, *args, **kwargs):
        if cls.metadata is None:
            cls._load_metadata()
        return super().__new__(cls)
    
    def __init__(self, *batch_names) -> None:
        self.data = CIFAR_10._read_file(batch_names[0])
        if len(batch_names) > 1:
            for batch_name in batch_names[1:]:
                self.data = CIFAR_10._merge_batches(self.data, CIFAR_10._read_file(batch_name))

        self.iter_idx = -1
            
    def __len__(self) -> int:
        return len(self.data['data'])

    def __getitem__(self, index_like):
        if isinstance(idx := index_like, int):
            return self.data['data'][idx], self.data['labels'][idx]
        raise ValueError(f'{index_like} is not an instance of int. currently other types are not supported.')

    def __iter__(self):
        return self

    def __next__(self):
        if self.iter_idx + 1 == len(self):
            raise StopIteration
        self.iter_idx += 1
        return self[iter_idx]
    
    def imshow(self, index) -> None:
        img = CIFAR_10._reconstruct_img(self.data['data'][index])
        label = metadata['label_names'][self.data['labels'][index]]
        
        plt.title(label)
        plt.imshow(img)
        plt.axis('off')

    @classmethod
    def _load_metadata(cls) -> None:
        cls.metadata = cls._read_file('batches.meta', 'ASCII')
    
    @staticmethod
    def _merge_batches(b1: dict, b2: dict) -> dict:
        merged_keys = ['batch_labels', 'labels', 'filenames', 'sizes']
        merged = {key: [] for key in merged_keys} 

        if 'batch_labels' in b1:
            for key in merged_keys:
                merged[key].extend(b1[key])
            merged['batch_labels'].append(b2['batch_label'])
            merged['labels'].extend(b2['labels'])
            merged['sizes'].append(len(b2['data']))
            merged['data'] = np.concat([b1['data'], b2['data']])
        else:
            merged['batch_labels'].append(b1['batch_label'])
            merged['batch_labels'].append(b2['batch_label'])
            
            merged['labels'].extend(b1['labels'])
            merged['labels'].extend(b2['labels'])
            
            merged['filenames'].extend(b1['filenames'])
            merged['filenames'].extend(b2['filenames'])

            merged['sizes'].append(len(b1['data']))
            merged['sizes'].append(len(b2['data']))

            merged['data'] = np.concat([b1['data'], b2['data']])

        return merged
    
    @staticmethod
    def _read_file(filename: str, encoding='bytes') -> dict:
        data = None
        with open(dataset_parent / filename, 'rb') as f:
            data = pickle.load(f, encoding=encoding)
        return data

    @staticmethod
    def _reconstruct_img(img_flatten: np.ndarray) -> np.ndarray:
        new_img = np.empty((32, 32, 3), np.uint8)
        new_img[:, :, 0] = img_flatten[0: 1 << 10].copy().reshape((32, 32))
        new_img[:, :, 1] = img_flatten[1 << 10: 1 << 11].copy().reshape((32, 32))
        new_img[:, :, 2] = img_flatten[1 << 11: 1 << 12].copy().reshape((32, 32))
        return new_img

In [3]:
dataset_parent = Path('datasets/cifar-10-batches-py')

In [4]:
train_batch = CIFAR_10('data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5')
test_batch = CIFAR_10('test_batch')

In [ ]:
class K_Nearest_Neighbors:
    def __init__(self, train_x, train_y, k):
        self.train_x = train_x
        self.train_y = train_y
        self.k = k
    
    def predict(self, x):
        if len(x.shape) == 1:
            return self._predict_one(x)
        else:
            res = []
            for _x in x:
                res.append(self._predict_one(_x))
            return np.array(res)
    
    def _predict_one(self, x):
        distances = self._compute_l1_distance(x)
        top_k = np.argsort(distances)[0: self.k]
        votes = np.bincount(top_k)
        majority = np.argmax(votes)
        return self.train_y[majority]

    def _compute_l1_distance(self, x):
        return np.sum(np.abs(self.train_x - x), axis=1)

    def _compute_l2_distance(self, x):
        return np.sum(np.sqrt(np.pow(self.train_x - x, 2), axis=1), axis=1)

In [6]:
knn = K_Neasrest_Neighbors(train_batch.data['data'], train_batch.data['labels'], 5)

In [7]:
%%time

test_x, test_y = test_batch.data['data'], test_batch.data['labels']
pred_y = knn.predict(test_x)

print('Accuracy:', np.mean(test_y == pred_y))

Accuracy: 0.225
CPU times: user 14min 56s, sys: 5min 38s, total: 20min 35s
Wall time: 20min 35s
